In [22]:
from langgraph.graph import StateGraph, START,END, MessagesState
from langchain_groq import ChatGroq
from dotenv import load_dotenv

from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages.utils import trim_messages, count_tokens_approximately
from langchain.messages import RemoveMessage

from typing import TypedDict, Annotated
from langchain_core.messages import HumanMessage, BaseMessage
from langgraph.graph.message import add_messages

In [23]:
load_dotenv()

True

In [ ]:
llm = ChatGroq(model='llama-3.3-70b-versatile')

In [25]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [ ]:
def chat(state: ChatState):
    messages = state['messages']
    response = llm.invoke(messages)
    
    return {
        'messages': [response]
    }
    
def delete_old_messages(state: ChatState):
    msgs = state["messages"]

    # if more than 10 messages, delete the start 6 not recent
    if len(msgs) > 10:
        to_remove = msgs[:6] 
        return {"messages": [RemoveMessage(id=m.id) for m in to_remove]}

    return {}

In [ ]:
builder = StateGraph(ChatState)
builder.add_node("chat", chat)
builder.add_node("cleanup", delete_old_messages)

NameError: name 'MessageState' is not defined

In [ ]:
builder.add_edge(START, "chat")
builder.add_edge("chat", "cleanup")   # run deletion after each response
builder.add_edge("cleanup", "__end__")

In [ ]:
checkpointer = InMemorySaver()
graph = graph.compile(checkpointer=checkpointer)

In [ ]:
config = {"configurable": {"thread_id": "t1"}}

In [ ]:
# Run multiple turns
graph.invoke({"messages": [{"role": "user", "content": "Hi, I'm Nitish"}]}, config)
graph.invoke({"messages": [{"role": "user", "content": "Tell me about LangGraph"}]}, config)
graph.invoke({"messages": [{"role": "user", "content": "Now explain checkpointers"}]}, config)
graph.invoke({"messages": [{"role": "user", "content": "What is Langchain"}]}, config)
graph.invoke({"messages": [{"role": "user", "content": "What is Quantum Mechanics"}]}, config)
graph.invoke({"messages": [{"role": "user", "content": "What is Gen AI"}]}, config)
graph.invoke({"messages": [{"role": "user", "content": "What is my name"}]}, config)